# Otimização de Prompts com GEPA + Kontex — Dados de Mineração

Este notebook demonstra o workflow de otimização de prompts usando o **GEPA**
(Genetic Pareto Prompt Optimizer) integrado ao **Kontex** com dados de
mineração **simulados via EDD** (Expertise Distribution Dynamics).

O agente otimizado é o `questioning`, cujo prompt de usuário é evoluído pelo
GEPA para que o agente faça perguntas mais eficazes aos especialistas,
resultando em descrições de tabelas mais completas.

## Visão geral

```
Simulação EDD (tema: mining)
→ especialistas com conhecimento distribuído sobre tabelas
        ↓
  Conversa multi-agente
  questioning → critique (prompt fixo)
        ↓
  GEvalMetric (acurácia factual + completude da descrição)
        ↓
  GEPA otimiza o prompt do questioning
```

---

**Diferença em relação ao notebook `00_00_basic_call.ipynb`:**
- Usa dados sintéticos gerados localmente (sem dataset externo)
- Otimiza o agente de **questionamento** (não o de construção de descrição)
- Usa `KontexFlow` (fluxo com crítico de prompt fixo) em vez de `KontexFlowGeneralized`

## 1. Configuração de paths e imports

In [ ]:
import sys
import os
from pathlib import Path

# Raiz do repositório kontex_gepa
root_dir = Path(os.getcwd()).parent.resolve()
parent_dir = root_dir.parent

sys.path.insert(0, str(root_dir))
sys.path.insert(0, str(parent_dir / "kontex" / "src"))
sys.path.insert(0, str(parent_dir / "gepa" / "src"))

os.environ["DATABASE_URL"] = f"sqlite:///{root_dir}/kontex_gepa_data.db"

print(f"root_dir : {root_dir}")
print(f"parent_dir: {parent_dir}")

## 2. Importações do módulo kontex_gepa

In [ ]:
from kontex_gepa import (
    EnvConfig,
    KontexFlow,
    GEvalMetric,
    GEVAL_CRITERIA_TABLE,
    generate_pareto_dataset,
)

from gepa import GEPAOptimizer, GEPAConfig
from gepa.core.system import CompoundAISystem, LanguageModule, IOSchema
from gepa.inference.factory import InferenceFactory
from gepa.config import InferenceConfig, OptimizationConfig, DatabaseConfig, ObservabilityConfig
from gepa.evaluation.base import SimpleFeedbackEvaluator

from kontex.simulation.edd.general_knowledge import FullKnowledge

print("✅ Importações concluídas.")

## 3. Variáveis de ambiente

In [ ]:
env = EnvConfig(env_file=str(root_dir / ".env"))

if not env.api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY não encontrado. "
        "Certifique-se de que o arquivo .env existe e contém a chave."
    )

print(f"API key: {'*' * 8}{env.api_key[-4:]}")
print(f"Base URL: {env.base_url}")
print(f"Model: {env.model}")

## 4. Geração do dataset de mineração via EDD

A simulação EDD distribui o conhecimento sobre uma tabela de mineração entre
vários especialistas, com esquecimento e conexões configuráveis. Cada
datapoint contém:
- `full_knowledge`: o conhecimento completo sobre a tabela
- `users_with_knowledge`: dicionário `{nome: Specialist}` com conhecimento parcial
- `question`: pergunta a ser respondida pelos agentes
- `expected_description`: descrição de referência para avaliação

In [ ]:
PARETO_SIZE = 1
FEEDBACK_SIZE = 1

dataset = generate_pareto_dataset(seed=42)

print(f"Dataset gerado com {len(dataset)} datapoints.")
print("\nExemplo (datapoint 0):")
dp = dataset[0]
print(f"  Questão  : {dp['question']}")
domain = list(dp['full_knowledge'].domains.keys())[0]
print(f"  Domínio  : {domain}")
print(f"  Colunas  : {list(dp['full_knowledge'].domains[domain].facts.keys())}")
print(f"  Especialistas: {list(dp['users_with_knowledge'].keys())}")

dataset = dataset[: PARETO_SIZE + FEEDBACK_SIZE]

## 5. Definição do CompoundAISystem

O módulo `questioning` é o alvo da otimização. Seu prompt de usuário guia
as perguntas feitas pelo agente aos especialistas. O crítico usa um prompt
fixo embutido em `KontexFlow`.

In [ ]:
INITIAL_QUESTIONING_PROMPT = """\
You're helping acquire knowledge about a table by questioning specialists.

Current Table Description:
{table_description}

Recent Critique:
{critique_response}

Conversation History with {specialist}:
{chat_history}

Generate a focused question for {specialist} to improve our table understanding.
Focus on:
- Column meanings and data types
- Example values
- Business context and relationships

This is very important: DO NOT ASK the specialist for SQL snippets to confirm data.
You only need the metadata, not the data itself.
Question:"""

system = CompoundAISystem(
    modules={
        "questioning": LanguageModule(
            id="questioning",
            prompt=INITIAL_QUESTIONING_PROMPT,
            model_weights="gpt-5-mini",
        )
    },
    control_flow=KontexFlow(),
    input_schema=IOSchema(
        fields={"full_knowledge": FullKnowledge},
        required=["full_knowledge"],
    ),
    output_schema=IOSchema(
        fields={"output": int},
        required=["output"],
    ),
    system_id="kontex_mining",
)

print("✅ CompoundAISystem criado.")
print(f"   Módulos: {list(system.modules.keys())}")
print(f"   Fluxo  : {type(system.control_flow).__name__}")

## 6. Configuração do GEPA

In [ ]:
gepa_config = GEPAConfig(
    inference=InferenceConfig(
        provider="openai",
        model="gpt-5-mini",
        api_key=env.api_key,
        max_tokens=4096,
        temperature=0.1,
        timeout=30,
        base_url=env.base_url,
        retry_attempts=3,
    ),
    optimization=OptimizationConfig(
        budget=20,
        pareto_set_size=PARETO_SIZE,
        minibatch_size=FEEDBACK_SIZE,
        enable_crossover=True,
        crossover_probability=0.3,
        mutation_types=["rewrite", "insert"],
    ),
    database=DatabaseConfig(url="sqlite:///gepa_mining.db"),
    observability=ObservabilityConfig(
        log_level="INFO",
        log_file="gepa_mining.log",
        enable_logging=True,
    ),
)

print("✅ GEPAConfig criado.")
print(f"   Budget      : {gepa_config.optimization.budget}")
print(f"   Pareto size : {gepa_config.optimization.pareto_set_size}")
print(f"   Mutations   : {gepa_config.optimization.mutation_types}")

## 7. Execução da otimização GEPA

## 7. Critérios de avaliação (GEval)

Edite as strings abaixo para personalizar como o `GEvalMetric` julga as descrições de tabela.
O padrão `GEVAL_CRITERIA_TABLE` já está carregado; basta sobrescrever se necessário.

In [ ]:
# Ponto de configuração: edite aqui para adaptar a avaliação ao seu experimento.
# Use GEVAL_CRITERIA_TABLE (descrição de tabelas) ou GEVAL_CRITERIA_HOTPOTQA
# (perguntas multi-hop), ou escreva critérios próprios.

FACTUAL_ACCURACY_CRITERIA = GEVAL_CRITERIA_TABLE["factual_accuracy"]
COMPLETENESS_CRITERIA     = GEVAL_CRITERIA_TABLE["completeness"]

print("Critério — Acurácia Factual:")
print(FACTUAL_ACCURACY_CRITERIA)
print()
print("Critério — Completude:")
print(COMPLETENESS_CRITERIA)

In [ ]:
import traceback

evaluator = SimpleFeedbackEvaluator([
    GEvalMetric(
        name="geval_metric",
        factual_accuracy_criteria=FACTUAL_ACCURACY_CRITERIA,
        completeness_criteria=COMPLETENESS_CRITERIA,
    )
])
inference_client = InferenceFactory.create_client(gepa_config.inference)

optimizer = GEPAOptimizer(
    config=gepa_config,
    evaluator=evaluator,
    inference_client=inference_client,
)

result = None
try:
    result = await optimizer.optimize(system, dataset, max_generations=5)
    print("✅ Otimização concluída!")
    print(f"   Melhor score    : {result.best_score:.3f}")
    print(f"   Total rollouts  : {result.total_rollouts}")
    print(f"   Custo total     : ${result.total_cost:.4f}")
    print(f"   Fronteira Pareto: {result.pareto_frontier.size()} pontos")
except Exception:
    print("❌ Otimização falhou:")
    print(traceback.format_exc())
finally:
    if hasattr(inference_client, "close"):
        await inference_client.close()

## 8. Resultados e salvamento do prompt otimizado

In [ ]:
if result is not None:
    from datetime import datetime
    best_module = result.best_system.modules["questioning"]
    print("🧠 Prompt de questionamento otimizado:")
    print("-" * 50)
    print(best_module.prompt)
    print("-" * 50)

    # Salva em disco com timestamp para não sobrescrever experimentos anteriores
    out_dir = root_dir / "optimized_prompts"
    out_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    out_file = out_dir / f"questioning_optimized_{timestamp}.txt"
    out_file.write_text(best_module.prompt)
    print(f"\n💾 Prompt salvo em: {out_file}")

    stats = optimizer.get_statistics()
    print("\n📊 Estatísticas:")
    print(f"   Gerações       : {stats.get('generations', 0)}")
    print(f"   Mutações ok    : {stats.get('successful_mutations', 0)}")
    print(f"   Melhoria média : {stats.get('average_improvement', 0):.3f}")
else:
    print("Nenhum resultado disponível (otimização falhou).")

## 9. (Opcional) Carregando dataset do CSV do Kontex

Em vez de gerar o dataset via simulação EDD, você pode carregar dados
pré-existentes do arquivo `simulated_table_info.csv` do repositório Kontex.

In [ ]:
import csv as csv_module
from uuid import UUID
from kontex_gepa import parse_users_info_to_specialists
from parse_full_knowledge import parse_full_knowledge_from_string
from kontex.simulation.edd.general_knowledge import FullKnowledge, DomainKnowledge

csv_path = parent_dir / "kontex" / "data" / "simulated_table_info.csv"

if csv_path.exists():
    csv_dataset = []
    with open(csv_path, encoding="utf-8") as f:
        for row in csv_module.DictReader(f):
            fk = parse_full_knowledge_from_string(
                row["full_knowledge"],
                FullKnowledge=FullKnowledge,
                DomainKnowledge=DomainKnowledge,
            )
            csv_dataset.append({
                "full_knowledge": fk,
                "run_id": UUID(row["run_id"]) if row.get("run_id") else None,
                "users_with_knowledge": parse_users_info_to_specialists(row["users_info"]),
                "question": row.get("question"),
                "expected": int(row.get("expected", 10)),
                "expected_description": row.get("expected_description"),
            })

    print(f"✅ CSV carregado: {len(csv_dataset)} datapoints.")
    print(f"   Exemplo: {csv_dataset[0]['question']}")
else:
    print(f"⚠ Arquivo não encontrado: {csv_path}")
    print("  Use generate_pareto_dataset() para gerar dados sintéticos.")